# 5e. Reconstruct matched-recall results and alert overlap

This notebook saves a reconstruction of two supporting analyses: the matched-recall M2–M0 comparison across detector families, and the overlap between the primary M0 and M2 alert sets.

**Inputs:** `scores_all_models.parquet` and the shared evaluation module `modelling_utils.py`, whose recorded SHA-256 prefix is `147d19b0977c79e8`.

The Isolation Forest matched-recall result belongs to the primary comparison. The autoencoder and ensemble matched-recall results are post-hoc sensitivity analyses.

Random-number order matters here. One seed-42 generator is shared across the detector loop in the order Isolation Forest, autoencoder and ensemble, with 300 bootstrap replicates per family.

**Outputs:** `posthoc_critA_by_detector.csv`, `alert_set_overlap.csv` and `provenance_5e.json`.

No detector is refitted. These checks concern matched recall and alert overlap, not a complete reproduction of every fixed-budget interval or upstream research stage.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
import os, glob, json, time, hashlib, warnings
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
WORK = "/kaggle/working"
K = 1.0
DET_ORDER = ["if", "ae", "ens"]     # the draw order used by the frozen NB5 loop
N_BOOT_MATCHED = 300
T0 = time.time()

def find(pattern, required=True, what=""):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits or not required, f"Required input not attached: {pattern} ({what})"
    return hits[0] if hits else None

P_SCORES = find("scores_all_models.parquet", what="NB5 committed scores")
P_UTILS  = find("modelling_utils.py",        what="NB4 shared logic")

# Import the SAME primitives the pipeline used, verbatim.
import importlib.util
spec = importlib.util.spec_from_file_location("modelling_utils", P_UTILS)
mu = importlib.util.module_from_spec(spec); spec.loader.exec_module(mu)
UTILS_SHA = hashlib.sha256(open(P_UTILS, "rb").read()).hexdigest()[:16]
assert UTILS_SHA == "147d19b0977c79e8", f"utils drift: {UTILS_SHA}"
print(f"modelling_utils.py sha256[:16] = {UTILS_SHA}  (matches frozen pipeline)")

scores_all = pd.read_parquet(P_SCORES)
test = scores_all[scores_all["partition"] == "test"].reset_index(drop=True)
yt = test["label"].to_numpy()
ut = test["user"].to_numpy()
print(f"test partition: {len(test):,} user-days | "
      f"{int(yt.sum())} labelled malicious from "
      f"{test.loc[yt.astype(bool), 'user'].nunique()} accounts")

modelling_utils.py sha256[:16] = 147d19b0977c79e8  (matches frozen pipeline)
test partition: 395,549 user-days | 38 labelled malicious from 2 accounts


## 1. Define the matched-recall helpers

The functions find the false-positive rate needed to reach the recall anchor and estimate its paired difference by resampling users. The same sampled accounts enter both configurations.

The detector loop supplies a shared random-number generator rather than starting a separate seed-42 stream for each family. This preserves the draw order specified for this reconstruction.


In [2]:
def fpr_at_recall(y, s, anchor):
    """Minimal FPR whose threshold achieves recall >= anchor."""
    y = np.asarray(y).astype(bool); s = np.asarray(s)
    P = int(y.sum())
    t = max(1, int(np.ceil(anchor * P - 1e-9)))
    pos = np.sort(s[y])[::-1]
    if t > len(pos):
        return np.nan, np.nan
    thr = pos[t - 1]
    return float((s[~y] >= thr).mean()), float((s[y] >= thr).sum() / P)

def paired_matched_fpr_delta(users, y, s_ref, s_new, anchor, rng, n_boot=N_BOOT_MATCHED):
    """Paired cluster bootstrap of Delta FPR at matched recall, drawing from a
    caller-supplied rng so the draw order matches the frozen NB5 loop."""
    df = pd.DataFrame({"user": users, "y": np.asarray(y).astype(int),
                       "a": s_ref, "b": s_new})
    groups = {u: (g["y"].to_numpy(), g["a"].to_numpy(), g["b"].to_numpy())
              for u, g in df.groupby("user")}
    upos = [u for u, (yy, _, _) in groups.items() if yy.any()]
    uben = [u for u, (yy, _, _) in groups.items() if not yy.any()]
    f_ref, _ = fpr_at_recall(df["y"], df["a"], anchor)
    f_new, _ = fpr_at_recall(df["y"], df["b"], anchor)
    dl = []
    for _ in range(n_boot):
        chosen = list(rng.choice(upos, len(upos))) + list(rng.choice(uben, len(uben)))
        yy = np.concatenate([groups[u][0] for u in chosen])
        sa = np.concatenate([groups[u][1] for u in chosen])
        sb = np.concatenate([groups[u][2] for u in chosen])
        fa, _ = fpr_at_recall(yy, sa, anchor)
        fb, _ = fpr_at_recall(yy, sb, anchor)
        if not (np.isnan(fa) or np.isnan(fb)):
            dl.append(fb - fa)
    lo, hi = np.percentile(dl, [2.5, 97.5])
    return {"point": f_new - f_ref, "ci_lo": float(lo), "ci_hi": float(hi),
            "fpr_ref": f_ref, "fpr_new": f_new, "n_used": len(dl)}
print("helpers defined")

helpers defined


## 2. Compare the reconstructed values with the saved targets

Recalculate Criterion A for Isolation Forest, the autoencoder and the ensemble, each using its own M0 recall anchor.

The recorded Isolation Forest target is `+0.00108 [-0.00022, +0.00348]`. Point estimates and interval endpoints are checked against rounded targets with tolerance `6e-5`; recall anchors use tolerance `1e-3`.

The assertion requires the Isolation Forest check to pass. Matching flags are also reported for the other families and should be reviewed individually. A pass is numerical agreement within the stated tolerance, not proof that every historical execution detail has been recovered.


In [3]:
COMMITTED = {
    "if":  {"point": +0.00108, "ci_lo": -0.00022, "ci_hi": +0.00348, "anchor": 0.1316},
    "ae":  {"point": -0.00128, "ci_lo": -0.00187, "ci_hi": +0.00153, "anchor": 0.1842},
    "ens": {"point": +0.00168, "ci_lo": -0.00073, "ci_hi": +0.00209, "anchor": 0.1579},
}
TOL = 6e-5   # tolerance on point/CI reproduction

rng = np.random.default_rng(SEED)          # SINGLE stream, shared across detectors
critA_rows = []
for det in DET_ORDER:
    s0 = test[f"M0_{det}"].to_numpy()
    s2 = test[f"M2_{det}"].to_numpy()
    anchor = mu.metrics_from_counts(*mu.counts(yt, mu.alerts_at_budget(s0, K)))["recall"]
    r = paired_matched_fpr_delta(ut, yt, s0, s2, anchor, rng)
    c = COMMITTED[det]
    ok = (abs(r["point"] - c["point"]) < TOL
          and abs(r["ci_lo"] - c["ci_lo"]) < TOL
          and abs(r["ci_hi"] - c["ci_hi"]) < TOL
          and abs(anchor - c["anchor"]) < 1e-3)
    critA_rows.append({"detector": det, "anchor_recall": anchor,
                       "dFPR_A": r["point"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"],
                       "matches_committed": bool(ok)})
    print(f"{det:4s} anchor={anchor:.4f}  dFPR_A={r['point']:+.5f} "
          f"[{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]  "
          f"{'OK' if ok else 'MISMATCH'}")

assert critA_rows[0]["matches_committed"], "Isolation Forest reproduction gate FAILED"
print("\nReproduction gate PASSED for the frozen Isolation Forest.")

if   anchor=0.1316  dFPR_A=+0.00108 [-0.00022, +0.00348]  OK
ae   anchor=0.1842  dFPR_A=-0.00128 [-0.00187, +0.00153]  OK
ens  anchor=0.1579  dFPR_A=+0.00168 [-0.00073, +0.00209]  OK

Reproduction gate PASSED for the frozen Isolation Forest.


## 3. Check the fixed-budget alert overlap

Use the primary Isolation Forest scores at the 1.0% budget to compare M0 and M2 membership directly.

The checks cover the 3,115 shared alerts, their 78.76% share of either set, and the 840 user-days unique to each configuration. Positive-label membership is checked on both sides, supporting the description of the changes as benign-for-benign substitutions rather than additional malicious-day detections.


In [4]:
a0 = mu.alerts_at_budget(test["M0_if"].to_numpy(), K)
a2 = mu.alerts_at_budget(test["M2_if"].to_numpy(), K)
n0, n2 = int(a0.sum()), int(a2.sum())
shared = int((a0 & a2).sum())
overlap = shared / n0
subs = n0 - shared
mal_m0_only = int(test.loc[a0 & ~a2, "label"].sum())
mal_m2_only = int(test.loc[~a0 & a2, "label"].sum())
mal_shared  = int(test.loc[a0 & a2, "label"].sum())

overlap_rec = {
    "budget_pct": K, "detector": "if",
    "n_alerts_m0": n0, "n_alerts_m2": n2,
    "shared_alerts": shared, "overlap_fraction": overlap,
    "substitutions": subs,
    "malicious_m0_only": mal_m0_only, "malicious_m2_only": mal_m2_only,
    "malicious_in_both": mal_shared,
    "benign_for_benign": bool(mal_m0_only == 0 and mal_m2_only == 0),
}
print(json.dumps(overlap_rec, indent=2))

# Assert against the committed Chapter 5 statement.
assert shared == 3115 and subs == 840 and abs(overlap - 0.7876) < 5e-4
assert mal_m0_only == 0 and mal_m2_only == 0 and mal_shared == 5
print("\nOverlap reproduction PASSED: 3,115 shared / 78.76% / 840 benign-for-benign.")

{
  "budget_pct": 1.0,
  "detector": "if",
  "n_alerts_m0": 3955,
  "n_alerts_m2": 3955,
  "shared_alerts": 3115,
  "overlap_fraction": 0.7876106194690266,
  "substitutions": 840,
  "malicious_m0_only": 0,
  "malicious_m2_only": 0,
  "malicious_in_both": 5,
  "benign_for_benign": true
}

Overlap reproduction PASSED: 3,115 shared / 78.76% / 840 benign-for-benign.


## 4. Save the reconstruction records

Write `posthoc_critA_by_detector.csv`, `alert_set_overlap.csv` and `provenance_5e.json`.

The JSON records the module identifier, budget, replicate count, random-number stream, detector order, comparison values and overlap checks. Notebook 6 can use the two CSVs for the corresponding reporting assets.


In [5]:
critA_df = pd.DataFrame(critA_rows)
critA_df["role"] = ["primary_frozen", "post_hoc_sensitivity", "post_hoc_sensitivity"]
critA_df.to_csv(f"{WORK}/posthoc_critA_by_detector.csv", index=False)

pd.DataFrame([overlap_rec]).to_csv(f"{WORK}/alert_set_overlap.csv", index=False)

provenance = {
    "utils_sha256_16": UTILS_SHA,
    "frozen_policy": {"detector": "if", "k_pct": K},
    "n_boot_matched": N_BOOT_MATCHED,
    "rng": {"type": "np.random.default_rng", "seed": SEED,
            "note": "single shared stream across detector order if, ae, ens"},
    "detector_order": DET_ORDER,
    "criterion_A": {r["detector"]: {k: r[k] for k in
                    ("anchor_recall", "dFPR_A", "ci_lo", "ci_hi", "matches_committed")}
                    for r in critA_rows},
    "alert_set_overlap": overlap_rec,
    "reproduction_gate_passed": True,
    "seed": SEED,
    "runtime_sec": round(time.time() - T0, 2),
}
json.dump(provenance, open(f"{WORK}/provenance_5e.json", "w"), indent=2, default=float)
print("wrote posthoc_critA_by_detector.csv, alert_set_overlap.csv, provenance_5e.json")
print(json.dumps(provenance, indent=2, default=float))

wrote posthoc_critA_by_detector.csv, alert_set_overlap.csv, provenance_5e.json
{
  "utils_sha256_16": "147d19b0977c79e8",
  "frozen_policy": {
    "detector": "if",
    "k_pct": 1.0
  },
  "n_boot_matched": 300,
  "rng": {
    "type": "np.random.default_rng",
    "seed": 42,
    "note": "single shared stream across detector order if, ae, ens"
  },
  "detector_order": [
    "if",
    "ae",
    "ens"
  ],
  "criterion_A": {
    "if": {
      "anchor_recall": 0.13157894736842105,
      "dFPR_A": 0.0010821443651377587,
      "ci_lo": -0.0002226270567991922,
      "ci_hi": 0.0034780179421395187,
      "matches_committed": true
    },
    "ae": {
      "anchor_recall": 0.18421052631578946,
      "dFPR_A": -0.0012844143399298614,
      "ci_lo": -0.0018661702631137952,
      "ci_hi": 0.0015319871026613305,
      "matches_committed": true
    },
    "ens": {
      "anchor_recall": 0.15789473684210525,
      "dFPR_A": 0.0016838975401442698,
      "ci_lo": -0.0007331490486995521,
      "ci_hi": 0

## 5. Read the result of this check

The saved outputs report the matched-recall comparisons and direct alert-overlap checks. Review the per-detector `matches_committed` values as well as the asserted Isolation Forest check.

The overlap record identifies the shared and configuration-specific user-days. The matched-recall record provides a rerunnable source for these particular sensitivity quantities. Neither is a claim that every number in the dissertation has been independently reproduced.

The detectors, scores and primary decision remain unchanged.
